In [ ]:
# Enhanced SwapDeep with Mouth Mask
!pip install -U insightface onnxruntime-gpu opencv-python requests tqdm torch gradio --quiet

import os
import requests
import cv2
import numpy as np
import gradio as gr
from insightface.app import FaceAnalysis
from insightface.model_zoo import get_model
from tqdm import tqdm

def download(url, path):
    if not os.path.isfile(path):
        print(f"Downloading {os.path.basename(path)}...")
        r = requests.get(url, stream=True)
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk: f.write(chunk)
        print("Done.")

# Download model
download("https://huggingface.co/countfloyd/deepfake/resolve/main/inswapper_128.onnx", "inswapper_128.onnx")

# Initialize models
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider']
face_analyser = FaceAnalysis(providers=providers)
face_analyser.prepare(ctx_id=0, det_size=(640, 640))
swapper = get_model("inswapper_128.onnx", providers=providers, download=False)

print("✅ Models loaded")

In [ ]:
def create_mouth_mask(face, frame):
    """Create enhanced mouth mask"""
    mask = np.zeros(frame.shape[:2], dtype=np.uint8)
    if not hasattr(face, 'landmark_2d_106') or face.landmark_2d_106 is None:
        return mask, None, (0,0,0,0)
    
    landmarks = face.landmark_2d_106
    if len(landmarks) < 106:
        return mask, None, (0,0,0,0)
    
    # Extended mouth region for eating/drinking
    mouth_indices = [65,66,62,70,69,18,19,20,21,22,23,24,0,8,7,6,5,4,3,2,
                    61,63,64,67,68,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86]
    
    valid_indices = [i for i in mouth_indices if i < len(landmarks)]
    mouth_landmarks = landmarks[valid_indices].astype(np.int32)
    
    # Create adaptive mask
    center = np.mean(mouth_landmarks, axis=0)
    mouth_height = np.max(mouth_landmarks[:, 1]) - np.min(mouth_landmarks[:, 1])
    mouth_width = np.max(mouth_landmarks[:, 0]) - np.min(mouth_landmarks[:, 0])
    
    # Expand for large movements
    expansion = 1.0 + (mouth_height / max(mouth_width, 1) * 0.3)
    expanded_landmarks = (mouth_landmarks - center) * expansion + center
    expanded_landmarks = np.clip(expanded_landmarks.astype(np.int32), 0, [frame.shape[1]-1, frame.shape[0]-1])
    
    # Multi-layer mask
    for scale in [1.2, 1.0, 0.8]:
        layer_landmarks = (expanded_landmarks - center) * scale + center
        hull = cv2.convexHull(layer_landmarks.astype(np.int32))
        cv2.fillConvexPoly(mask, hull, 255)
    
    # Blur for natural blending
    mask = cv2.GaussianBlur(mask, (31, 31), 10)
    mask = cv2.GaussianBlur(mask, (15, 15), 5)
    
    # Bounding box
    x, y, w, h = cv2.boundingRect(expanded_landmarks)
    padding = 15
    x, y = max(0, x-padding), max(0, y-padding)
    w, h = min(frame.shape[1]-x, w+2*padding), min(frame.shape[0]-y, h+2*padding)
    
    mouth_cutout = frame[y:y+h, x:x+w].copy() if w > 0 and h > 0 else None
    return mask, mouth_cutout, (x, y, x+w, y+h)

def enhanced_face_swap(source_img, target_img, use_mouth_mask=True):
    """Enhanced face swap with mouth mask"""
    if source_img is None or target_img is None:
        return None, "Upload both images"
    
    # Convert RGB to BGR
    source = cv2.cvtColor(source_img, cv2.COLOR_RGB2BGR)
    target = cv2.cvtColor(target_img, cv2.COLOR_RGB2BGR)
    
    # Get faces
    source_faces = face_analyser.get(source)
    target_faces = face_analyser.get(target)
    
    if not source_faces:
        return target_img, "❌ No face in source"
    if not target_faces:
        return target_img, "❌ No face in target"
    
    # Face swap
    result = swapper.get(target, target_faces[0], source_faces[0], paste_back=True)
    
    # Apply mouth mask
    if use_mouth_mask:
        mask, mouth_cutout, box = create_mouth_mask(target_faces[0], target)
        if mouth_cutout is not None and box != (0,0,0,0):
            x1, y1, x2, y2 = box
            roi = result[y1:y2, x1:x2]
            
            if roi.shape[:2] == mouth_cutout.shape[:2]:
                mask_roi = mask[y1:y2, x1:x2] / 255.0
                blended = mouth_cutout * mask_roi[:,:,None] + roi * (1 - mask_roi[:,:,None])
                result[y1:y2, x1:x2] = blended.astype(np.uint8)
    
    # Convert back to RGB
    result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    return result_rgb, "✅ Success with mouth mask!"

print("✅ Enhanced face swap ready")

In [ ]:
# Launch Gradio interface
with gr.Blocks(title="Enhanced SwapDeep") as demo:
    gr.Markdown("# 🚀 Enhanced SwapDeep with Mouth Mask")
    gr.Markdown("Perfect for eating, drinking, and large mouth movements")
    
    with gr.Row():
        with gr.Column():
            source_input = gr.Image(label="📷 Source Face", type="numpy")
            target_input = gr.Image(label="🎯 Target Image", type="numpy")
            mouth_mask_check = gr.Checkbox(label="🦷 Enable Mouth Mask", value=True)
            swap_button = gr.Button("🔄 Swap Faces", variant="primary")
        
        with gr.Column():
            result_output = gr.Image(label="✨ Result")
            status_output = gr.Textbox(label="📊 Status")
    
    gr.Markdown("""
    ### 📋 Instructions:
    1. Upload source face (clear, front-facing)
    2. Upload target image
    3. Enable mouth mask for natural results
    4. Click Swap Faces
    
    **Mouth Mask Benefits:**
    - Preserves natural mouth/teeth
    - Perfect for eating/drinking scenarios
    - Handles large mouth movements
    """)
    
    swap_button.click(
        enhanced_face_swap,
        inputs=[source_input, target_input, mouth_mask_check],
        outputs=[result_output, status_output]
    )

demo.launch(share=True, debug=True)